In [ ]:
!nvidia-smi
%env CUDA_VISIBLE_DEVICES=0

In [ ]:
import glob, os, shutil, subprocess
if not os.path.isdir("privify-aiact-tier"):
    pkgs = glob.glob("/kaggle/input/**/aiact_tier", recursive=True)
    zips = glob.glob("/kaggle/input/**/privify-aiact-tier*.zip", recursive=True)
    if pkgs:
        shutil.copytree(os.path.dirname(pkgs[0]), "privify-aiact-tier")
    elif zips:
        subprocess.run(["unzip", "-q", "-o", zips[0]], check=True)
    else:
        raise SystemExit("Repo not found: attach the dataset under Input")
cfg = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
if not cfg:
    raise SystemExit("adapter_config.json not found: add it to the dataset")
ad = os.path.dirname(cfg[0])
os.makedirs("privify-aiact-tier/outputs", exist_ok=True)
if not os.path.isdir("privify-aiact-tier/outputs/aiact-tier-lora"):
    shutil.copytree(ad, "privify-aiact-tier/outputs/aiact-tier-lora")
print(os.listdir("privify-aiact-tier/outputs/aiact-tier-lora"))

In [ ]:
%cd privify-aiact-tier
!pip install -q -r requirements.txt pytest
!pip uninstall -y -q torchao
!python -m pytest -q

In [ ]:
!make data

In [ ]:
!make eval-zeroshot

In [ ]:
!make eval-ours

In [ ]:
!mkdir -p pkg && cp outputs/aiact-tier-lora/train_meta.json pkg/ 2>/dev/null; cp -r eval/results pkg/results
!cd pkg && zip -qr ../results.zip . && ls -la ../results.zip